In [2]:
import numpy as np
import matplotlib.pyplot as plt

In [3]:
I_1 = 0.1 # [kg.m^2]
I_2 = 2 # [kg.m^2]
c = 10 # [N.m/rad]

In [4]:
class SDOFTorsionalSystem:

    # Constructor to initialize the system parameters
    def __init__(self, inertia=0, stiffness=0):
        self.I = inertia
        self.c = stiffness
        if self.I > 0 and self.c > 0:
            self.omega_n = self.calculate_natural_frequency()
        else:
            self.omega_n = None

    # Method to calculate the natural frequency of the system
    def calculate_natural_frequency(self):
        return np.sqrt(self.c / self.I)

    # Method to simulate the time response of the system
    # NOTE: modified to account for any harmonic excitation
    def simulate_time_response(self, initial_angle=0, initial_velocity=0, time_span=10, load_type="poly", tau_ext=np.array([0], dtype=np.float16)):
        if self.omega_n is None:
            raise ValueError("Natural frequency is not defined. Please set inertia and stiffness.")
        
        if load_type == "measurement":
            # Time array
            t = tau_ext['time']
            # Torque array
            tau_ti = tau_ext['load']
            # Initialize array for angular displacement to zeros
            theta_t = np.zeros_like(t)
            # Initial conditions
            theta_t[0] = initial_angle
            theta_dot_i = initial_velocity
            # Iteratively calculate the angular displacement at each time step
            for i in range(len(t)-1):
                # Calculate the angular acceleration using the equation of motion
                theta_double_dot = (tau_ti[i] - self.c * theta_t[i]) / self.I
                # Update angular velocity and displacement using finite difference approximation
                dt = t[i+1] - t[i]
                theta_dot_i = theta_dot_i + theta_double_dot * dt
                theta_t[i+1] = theta_t[i] + theta_dot_i * dt
        else:
            # Time array
            t = np.linspace(0, time_span, 1000)
            # Natural frequency
            omega_n = self.omega_n
            if load_type == "poly":
                n = len(tau_ext) - 1 # Polynomial Order
                thetaP_i = np.zeros_like(tau_ext)
                thetaP_t = np.zeros_like(t)
                for i, tau_i in enumerate(tau_ext):
                    if i < 2:
                        thetaP_i[i] = tau_i / self.c # Equation of coefficient for i in [0, 1]
                    else:
                        thetaP_i[i] = (tau_i - self.I*(n-i+2)*(n-i+1)*thetaP_i[i-2]) / self.c # Equation of coefficient for i in [2, n]
                    thetaP_t = thetaP_t + thetaP_i[i] * t ** (n-i) # Equation of particular solution as summation of
                thetaP_0 = thetaP_i[-1] # Constant term yields initital condition of theta
                if n > 0:
                    thetaPdot_0 = thetaP_i[-2] # Linear term yields initial condition of theta_dot
                else:
                    thetaPdot_0 = 0 # For free vibrations or constant external load
            elif load_type == "harmonic":
                # Initialize Particular Solution
                thetaP_t = np.zeros_like(t)
                # Sin terms
                if 'sin' in tau_ext:
                    A_s = tau_ext['sin']['amplitude']
                    omega_s = tau_ext['sin']['frequency'] * 2 * np.pi # Convert frequency from Hz to rad/s
                    if len(A_s) != len(omega_s):
                        raise Exception("Sin terms: amplitudes and freuencies must have the same length.")
                    theta_s_i = np.zeros_like(A_s) # Sin terms of particular solution initialization
                    for i, (A_i, omega_i) in enumerate(zip(A_s, omega_s)):
                        theta_s_i[i] = A_i / (self.c - self.I * omega_i ** 2) # Apply derived formula
                        thetaP_t = thetaP_t + theta_s_i[i] * np.sin(omega_i * t) # Add sin terms
                    thetaPdot_0 = np.sum(theta_s_i)  # Initial value of the derivative of the particular solution at t=0
                else:
                    thetaPdot_0 = 0
                # Cos terms
                if 'cos' in tau_ext:
                    B_c = tau_ext['cos']['amplitude']
                    omega_c = tau_ext['cos']['frequency'] * 2 * np.pi # Convert frequency from Hz to rad/s
                    if len(B_c) != len(omega_c):
                        raise Exception("Cos terms: amplitudes and freuencies must have the same length.")
                    theta_c_j = np.zeros_like(B_c) # Cos terms of particular solution initialization
                    for j, (B_j, omega_j) in enumerate(zip(B_c, omega_c)):
                        theta_c_j[j] = B_j / (self.c - self.I * omega_j ** 2) # Apply derived formula
                        thetaP_t = thetaP_t + theta_c_j[j] * np.cos(omega_j * t) # Add cos terms
                    thetaP_0 = np.sum(theta_c_j)  # Initial value of the particular solution at t=0
                else:
                    thetaP_0 = 0
            else:
                raise ValueError("Unsupported load type. Please specify 'constant' for constant external torque, 'linear' for linear torque ramp, or 'harmonic' for harmonic excitation.")

            # Calculate the constants A and B based on initial conditions
            A = initial_angle - thetaP_0  # Adjusting for the particular solution at t=0
            B = (initial_velocity - thetaPdot_0) / omega_n  # Adjusting for the derivative of the particular solution at t=0
            # General solution for the homogeneous part
            thetaG_t = A * np.cos(omega_n * t) + B * np.sin(omega_n * t)
            # Time response using the analytical solution for the specified load type
            theta_t = thetaG_t + thetaP_t
        return t, theta_t

In [15]:
class TwoDOFTorsionalSystem:

    # Constructor to initialize the system parameters
    def __init__(self, I_1=0, I_2=0, c=0):
        self.I_1 = I_1
        self.I_2 = I_2
        self.c = c
        if self.I_1>0 and self.I_2>0 and self.c>0:
            self.I_eq = (self.I_1 * self.I_2) / (self.I_1 + self.I_2)
            self.s_eq = SDOFTorsionalSystem(inertia=self.I_eq, stiffness=c)
            self.omega_n = self.s_eq.calculate_natural_frequency()
        else:
            self.omega_n = None
    
    # Method to simulate the time response of the system
    def simulate_time_response(self, initial_angle=0, initial_velocity=0, time_span=10, load_type="poly", tau_ext=np.array([0], dtype=np.float16)):
        t = np.zeros((int(time_span*1e3), 2))
        theta_t = np.zeros((int(time_span*1e3), 2))
        for i in range(2):
            t[:, i], theta_t[:, i] = self.s_eq.simulate_time_response(initial_angle=initial_angle[i], initial_velocity=initial_velocity[i], time_span=time_span, load_type=load_type[i], tau_ext=tau_ext[i])
        return t, theta_t

In [16]:
S = TwoDOFTorsionalSystem(I_1=I_1, I_2=I_2, c=c)
S.omega_n

np.float64(10.246950765959598)

In [17]:
theta_0 = [1, 0]
theta_dot_0 = [0, 0]
t, theta_t = S.simulate_time_response(initial_angle=theta_0, initial_velocity=theta_dot_0)

ValueError: Unsupported load type. Please specify 'constant' for constant external torque, 'linear' for linear torque ramp, or 'harmonic' for harmonic excitation.